# Spectrum Analysis: Cosine & Square Waves

*Python/Colab conversion of the MATLAB scripts `speccos.m` and `specsquare.m`*
(from **Software Receiver Design**, Johnson, Sethares & Klein).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/spectrum_analysis.ipynb)

> **How to run:** Click the **Open in Colab** badge above, then run each cell top-to-bottom with `Shift + Enter`, or use **Runtime -> Run all**.

This notebook shows each spectrum in three ways:
1. **Full view** 
2. **Zoomed-in view** around the spikes (static)
3. **Interactive view** — drag-to-zoom, box-zoom, pan and hover (Plotly)

No installation is needed — `numpy`, `matplotlib` and `plotly` are pre-installed in Colab.

## 1. Setup

Import the numerical, plotting and interactive-plotting libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Interactive plotting (pre-installed in Colab). Install locally if missing.
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'])
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

# Make static plots appear inline and a little larger
%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 6)

## 2. The `plotspec` helper function (with optional zoom)

A helper from the textbook that plots both the **time-domain waveform** and its **magnitude spectrum** (via the FFT).
It is not part of standard Python, so we re-implement it below.

We add one extra optional argument, **`flim`**, to zoom the spectrum:
* `plotspec(x, Ts)` → full spectrum (original behaviour)
* `plotspec(x, Ts, flim=60)` → zoom the frequency axis to ±60 Hz
* `plotspec(x, Ts, flim=(0, 60))` → zoom to a custom range 0–60 Hz

In [ ]:
def plotspec(x, Ts, flim=None):
    """Plot the waveform and magnitude spectrum of signal x.

    Parameters
    ----------
    x    : 1-D array of signal samples.
    Ts   : sample interval in seconds (Ts = 1/sample_rate).
    flim : optional frequency-axis zoom for the spectrum.
           * None            -> show full spectrum (default)
           * a number f      -> zoom to (-f, +f) Hz
           * a tuple (lo, hi)-> zoom to (lo, hi) Hz
    """
    x = np.asarray(x)
    N = len(x)                              # length of the signal
    t = Ts * np.arange(1, N + 1)            # time vector
    ssf = np.arange(-N/2, N/2) / (Ts * N)   # frequency vector (Hz)
    fx = np.fft.fft(x)                      # DFT / FFT
    fxs = np.fft.fftshift(fx)              # shift zero-freq to center

    fig, (ax1, ax2) = plt.subplots(2, 1)
    ax1.plot(t, x)                         # time-domain waveform
    ax1.set_xlabel('seconds')
    ax1.set_ylabel('amplitude')
    ax1.set_title('Waveform (time domain)')
    ax1.grid(True)

    ax2.plot(ssf, np.abs(fxs))             # magnitude spectrum
    ax2.set_xlabel('frequency (Hz)')
    ax2.set_ylabel('magnitude')
    ax2.grid(True)

    # Apply optional zoom on the frequency axis
    if flim is not None:
        if np.isscalar(flim):
            ax2.set_xlim(-flim, flim)
            ax2.set_title(f'Magnitude spectrum (zoomed to ±{flim} Hz)')
        else:
            ax2.set_xlim(flim[0], flim[1])
            ax2.set_title(f'Magnitude spectrum (zoomed to {flim[0]}–{flim[1]} Hz)')
    else:
        ax2.set_title('Magnitude spectrum (full)')

    fig.tight_layout()
    plt.show()

## 3. Interactive version: `plotspec_interactive`

This Plotly version lets you **zoom the plot themselves** — no code changes needed:

- **Box-zoom:** click and drag a rectangle over any region
- **Pan:** switch to the pan tool (top-right toolbar) and drag
- **Hover:** point at a spike to read its exact frequency & magnitude
- **Reset:** double-click, or use the 'Autoscale' / home button

You can still pass `flim` to set the initial zoom of the spectrum.

In [ ]:
def plotspec_interactive(x, Ts, flim=None, title=''):
    """Interactive (drag-to-zoom) waveform + magnitude spectrum using Plotly."""
    x = np.asarray(x)
    N = len(x)
    t = Ts * np.arange(1, N + 1)
    ssf = np.arange(-N/2, N/2) / (Ts * N)
    fxs = np.fft.fftshift(np.fft.fft(x))

    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=('Waveform (time domain)',
                                        'Magnitude spectrum'))
    fig.add_trace(go.Scatter(x=t, y=x, mode='lines', name='waveform'),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=ssf, y=np.abs(fxs), mode='lines',
                             name='|spectrum|'), row=2, col=1)

    fig.update_xaxes(title_text='seconds', row=1, col=1)
    fig.update_yaxes(title_text='amplitude', row=1, col=1)
    fig.update_xaxes(title_text='frequency (Hz)', row=2, col=1)
    fig.update_yaxes(title_text='magnitude', row=2, col=1)

    # Optional initial zoom on the frequency axis
    if flim is not None:
        lo, hi = (-flim, flim) if np.isscalar(flim) else (flim[0], flim[1])
        fig.update_xaxes(range=[lo, hi], row=2, col=1)

    fig.update_layout(height=650, showlegend=False, title_text=title,
                      hovermode='x unified')
    fig.show()

## 4. `speccos` — Spectrum of a cosine wave

A pure cosine at 10 Hz produces two spectral spikes at **+10 Hz** and **-10 Hz**.

In [ ]:
# speccos: plot the spectrum of a cosine wave
f = 10          # frequency (Hz)
phi = 0         # phase
time = 2        # length of time (seconds)
Ts = 1/100      # time interval between samples

t = np.arange(Ts, time + Ts, Ts)      # time vector: Ts:Ts:time
x_cos = np.cos(2*np.pi*f*t + phi)     # create cosine wave

plotspec(x_cos, Ts)                   # full waveform and spectrum

**Zoomed-in view** — close-up around the ±10 Hz spikes:

In [ ]:
# Static zoom to +/- 25 Hz to see the two spikes clearly
plotspec(x_cos, Ts, flim=25)

**Interactive view** — drag a box over the spikes to zoom, hover to read values:

In [ ]:
plotspec_interactive(x_cos, Ts, flim=25, title='Cosine wave (10 Hz)')

## 5. `specsquare` — Spectrum of a square wave

A square wave contains the fundamental at 10 Hz **plus odd harmonics** (30 Hz, 50 Hz, 70 Hz, ...), which you'll see as a series of spikes.

In [ ]:
# specsquare: plot the spectrum of a square wave
f = 10          # "frequency" of square wave (Hz)
time = 2        # length of time (seconds)
Ts = 1/1000     # time interval between samples

t = np.arange(Ts, time + Ts, Ts)      # time vector: Ts:Ts:time
x_sq = np.sign(np.cos(2*np.pi*f*t))   # square wave = sign of cosine

plotspec(x_sq, Ts)                    # full waveform and spectrum

**Zoomed-in view** — close-up on 0–100 Hz to reveal the odd harmonics (10, 30, 50, 70, 90 Hz):

In [ ]:
# Static zoom to 0-100 Hz to see the fundamental + odd harmonics
plotspec(x_sq, Ts, flim=(0, 100))

**Interactive view** — zoom/pan across the harmonics and hover to read each one:

In [ ]:
plotspec_interactive(x_sq, Ts, flim=(0, 100), title='Square wave (10 Hz)')

## 6. Notes & tips

- **Zooming statically:** pass `flim` to `plotspec`, e.g. `plotspec(x, Ts, flim=25)` for ±25 Hz, or `plotspec(x, Ts, flim=(0, 100))` for a custom range.
- **Zooming interactively:** use `plotspec_interactive(...)` and drag a box over any region; double-click to reset. Great for exploring the harmonics live.
- **Try it yourself:** change `f`, `phi`, or `Ts` and re-run to see how the spectrum reacts.

